# Check MSE and r^2 per participant

## Set Up

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH

import pandas as pd

In [ ]:
metrics_dir = BASE_PATH / "initial_results"
preds_dir = BASE_PATH / "initial_preds"

## Funcs

In [ ]:
import matplotlib.pyplot as plt


def get_mse_by_participant(loop_dir, agg_type):
    df_list = []
    for repeat_dir in loop_dir.iterdir():
        for split_path in repeat_dir.iterdir():
            split_file = pd.read_parquet(split_path)
            split_file["squared_error"] = (split_file["pred"] - split_file["true"]) ** 2
            df_list.append(split_file[["DAWBA_ID", "squared_error"]])

    all_rows = pd.concat(df_list, ignore_index=True)
    avg_error_by_id = (
        all_rows.groupby("DAWBA_ID", as_index=False)["squared_error"]
        .mean()
        .rename(columns={"squared_error": f"avg_error_{agg_type}"})
    )
    avg_error_by_id = avg_error_by_id.set_index("DAWBA_ID")
    return avg_error_by_id


def get_best_params(impact_type, agg_type, optimize="mse"):
    reduced_condition_dict = {
        "impact_type": impact_type,
    }
    if agg_type == "reduced":
        reduced_metrics_df = pd.read_csv(
            metrics_dir / f"params_optimize_{optimize}" / f"reduced.tsv",
            index_col=0,
            sep="\t",
        )
        reduced_metrics_df_sub = reduced_metrics_df.loc[
            (
                reduced_metrics_df[list(reduced_condition_dict)]
                == pd.Series(reduced_condition_dict)
            ).all(axis=1)
        ]
        l1_val = reduced_metrics_df_sub["l1"].item()
        alpha_val = reduced_metrics_df_sub["alpha"].item()
        return {
            "l1": l1_val,
            "alpha": alpha_val,
        }
    else:
        full_condition_dict = {
            **reduced_condition_dict,
            "agg_type": agg_type,
            # 'seg_type': seg_type,
            # 'model_name': model_name
        }
        # Get metrics df
        full_metrics_df = pd.read_csv(
            metrics_dir / f"params_optimize_{optimize}" / f"full.tsv",
            index_col=0,
            sep="\t",
            skipinitialspace=True,
        )
        ## Get row of metrics df specifying best score w/ current impact_type & agg_type
        full_metrics_df_sub = full_metrics_df.loc[
            (
                full_metrics_df[list(full_condition_dict)]
                == pd.Series(full_condition_dict)
            ).all(axis=1)
        ]
        # Ensure sorted & get best row
        best_row = full_metrics_df_sub.sort_values(
            by=f"avg_{optimize}", ascending=True
        ).iloc[0]
        ## Return row as dict
        return best_row.to_dict()


def plot_fig(df, impact_type, agg_types, sort_type):
    df_join_reset = df.reset_index()
    plt.figure(figsize=(14, 6))
    plt.plot(
        df_join_reset.index,
        df_join_reset["avg_error_reduced"],
        label="avg_error_reduced",
        marker="o",
        markersize=3,
    )
    for agg_type in agg_types:
        label = f"avg_error_{agg_type}"
        plt.plot(
            df_join_reset.index,
            df_join_reset[label],
            label=label,
            marker="o",
            markersize=3,
        )
        plt.title(
            f"Subject type: {impact_type.upper()}\n Aggregation type: {agg_type.upper()}\nAverage squared error accross 10-repeated 4-fold CV per participant"
        )
    plt.legend()
    plt.xlabel(f"Sorted on: {sort_type}")
    plt.ylabel("MSE")

    plt.show()

## Run

In [ ]:
agg_types = ["max", "avg", "block", "0_shot"]
df_dict = {}
for impact_type in ["self", "parent"]:
    ## Reduced params
    params = get_best_params(impact_type=impact_type, agg_type="reduced")
    alpha_val, l1_val = params["alpha"], params["l1"]
    ## Loop through preds and get avg metrics per patient
    loop_dir = (
        preds_dir / "reduced" / impact_type / f"alpha_{alpha_val}" / f"l1_{l1_val}"
    )
    reduced_avg_error_by_id = get_mse_by_participant(
        loop_dir=loop_dir, agg_type="reduced"
    )
    df_join = reduced_avg_error_by_id.copy()
    ### Full params
    for agg_type in agg_types:
        params = get_best_params(impact_type=impact_type, agg_type=agg_type)
        alpha_val = params["alpha"]
        l1_val = params["l1"]
        model_name = params["model_name"]
        if agg_type == "0_shot":
            desc_type = params["desc_type"]
            output_type = params["output_type"]
            loop_dir = (
                preds_dir
                / agg_type
                / impact_type
                / model_name
                / f"output_{output_type}"
                / f"desc_{desc_type}"
                / f"alpha_{alpha_val}"
                / f"l1_{l1_val}"
            )
        else:
            seg_type = params["seg_type"]
            loop_dir = (
                preds_dir
                / agg_type
                / impact_type
                / model_name
                / seg_type
                / f"alpha_{alpha_val}"
                / f"l1_{l1_val}"
            )
        cur_avg_error = get_mse_by_participant(loop_dir=loop_dir, agg_type=agg_type)
        df_join = df_join.join(cur_avg_error)
        df_join[f"mse_decrease_{agg_type}"] = (
            df_join["avg_error_reduced"] - df_join[f"avg_error_{agg_type}"]
        )
    # df_join = df_join.sort_values(by="avg_error_reduced", ascending=True)
    df_dict[impact_type] = df_join

Plot

In [ ]:
agg_of_interest = ["0_shot", "max"]
for impact_type in ["self", "parent"]:
    cur_df = df_dict[impact_type]
    ## Sort on baseline (reduced MSE)
    df_reduced_sorted = cur_df.sort_values(by=f"avg_error_reduced", ascending=False)
    for agg_type in agg_of_interest:
        ## Plot sorted by baseline MSEs
        plot_fig(
            df=df_reduced_sorted,
            impact_type=impact_type,
            agg_types=[agg_type],
            sort_type="Baseline MSE",
        )
        ## Plot sorted by decrease in MSE
        cur_df_sorted = cur_df.sort_values(
            by=f"mse_decrease_{agg_type}", ascending=False
        )
        plot_fig(
            df=cur_df_sorted,
            impact_type=impact_type,
            agg_types=[agg_type],
            sort_type=f"Diff in avg MSE between reduced & {agg_type} model",
        )
        ## Plot histogram of differences
        plt.hist(
            df_reduced_sorted[f"mse_decrease_{agg_type}"], bins=5, edgecolor="black"
        )

## Look at DAWBA IDs

In [ ]:
df_dir = BASE_PATH / "data" / "cleaned" / "processed"
parent_df_import = pd.read_parquet(df_dir / "parent.parquet")
parent_df_import.groupby(by="no_text").impact_score_parent.describe()

In [ ]:
cur_df = df_dict["parent"].sort_values(by="mse_decrease_0_shot", ascending=False)
cur_df.reset_index(inplace=True, drop=False)
## Meaningful decrease is ~ >=20% decrease (looks like all decrease are >= 20%)
cur_df["mse_decrease_0_perc"] = 100 * (
    cur_df["mse_decrease_0_shot"] / cur_df["avg_error_reduced"]
)

# helped_df = cur_df[cur_df['mse_decrease_0_perc'] > 20]
helped_df = cur_df.iloc[:5]
not_helped_df = cur_df.iloc[5:]

helped_ids = helped_df["DAWBA_ID"].to_list()
not_helped_ids = not_helped_df["DAWBA_ID"].to_list()

In [ ]:
original_df_import = pd.read_csv(
    BASE_PATH / "data" / "complete_raw" / "comppsych_dawbas_20210414-0.csv", index_col=0
)
## Participants who had large improvement in MSE
og_df_interest = original_df_import[original_df_import["DAWBA_ID"].isin(helped_ids)]
## All other participants
og_df_other = original_df_import[original_df_import["DAWBA_ID"].isin(not_helped_ids)]

In [ ]:
df_dir = BASE_PATH / "data" / "cleaned" / "processed"
parent_df_import = pd.read_parquet(df_dir / "parent.parquet")
## Participants who had large improvement in MSE
parent_df_interest = parent_df_import[parent_df_import["DAWBA_ID"].isin(helped_ids)]
## All other participants
other_parent_df = parent_df_import[~parent_df_import["DAWBA_ID"].isin(helped_ids)]

Compare groups

In [ ]:
print(parent_df_interest["eligible_group"].value_counts(normalize=True))
print(parent_df_interest["no_text"].value_counts(normalize=True))
print(parent_df_interest["Depression_Check_parent"].value_counts(normalize=True))
print(parent_df_interest["Irritable_Check_parent"].value_counts(normalize=True))
print(parent_df_interest["Loss_Interest_Check_parent"].value_counts(normalize=True))
print(parent_df_interest["impact_score_parent"].describe())
print(og_df_interest["DAWBA_SEX"].value_counts(normalize=True))
print(og_df_interest["DAWBA_Age_at_visit"].describe())

In [ ]:
shot_df = pd.read_parquet(
    "/home/mataray/DAWBA/data/w_0_shot_probs/temp_0.6/proba-limited/Llama-3.3-70B-Instruct/augment/parent.parquet"
)
df_sub = shot_df[shot_df["DAWBA_ID"].isin(helped_ids)]
for row in df_sub[["open_text_block", "open_text_block_0_shot"]].itertuples():
    print(row.open_text_block)
    print(row.open_text_block_0_shot)
    print("*" * 50)

Not helped

In [ ]:
print(other_parent_df["eligible_group"].value_counts(normalize=True))
print(other_parent_df["no_text"].value_counts(normalize=True))
print(other_parent_df["Depression_Check_parent"].value_counts(normalize=True))
print(other_parent_df["Irritable_Check_parent"].value_counts(normalize=True))
print(other_parent_df["Loss_Interest_Check_parent"].value_counts(normalize=True))
print(other_parent_df["impact_score_parent"].describe())
print(og_df_other["DAWBA_SEX"].value_counts(normalize=True))
print(og_df_other["DAWBA_Age_at_visit"].describe())